# Exploración del padrón de beneficiarios

**Objetivo:** entender qué hay en el archivo y hacer una lista de los problemas
que el ETL va a tener que resolver.

Esta notebook **no transforma nada**: solo mira. La transformación vive en `src/`.
La notebook explora, el script produce.

> **Regla de oro para explorar:** leer todo como texto (`dtype=str`). Si dejamos que
> pandas adivine los tipos, va a "arreglar" cosas por su cuenta y no vamos a ver la
> suciedad que estamos buscando.

In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", None)

# dtype=str          -> todo como texto, sin conversiones automáticas
# keep_default_na="" -> los vacíos quedan como "", no como NaN
df = pd.read_csv(
    "../data/raw/padron_beneficiarios.csv",
    dtype=str,
    keep_default_na=False,
)

print(f"{df.shape[0]} filas, {df.shape[1]} columnas")
df.head()

---
## 1. ¿Qué columnas hay?

In [ ]:
df.info()

Todas las columnas son `object`, es decir texto. **Eso no es un error de pandas: es
lo que es un CSV.** Un CSV no guarda tipos, así que la primera tarea del ETL va a ser
convertir.

---
## 2. ¿Qué falta?

Buscamos celdas vacías. Ojo: un dato faltante también puede venir escrito, como
`"s/d"`. Esos no aparecen como vacíos y hay que buscarlos aparte.

In [ ]:
vacios = (df == "").sum()
vacios[vacios > 0]

---
## 3. La columna `edad`

Debería ser un número. Veamos cuántos valores **no** lo son.

In [ ]:
no_numericas = df.loc[~df["edad"].str.strip().str.isdigit(), "edad"]

print(f"{len(no_numericas)} valores de edad no son un número limpio\n")
no_numericas.value_counts().head(8)

Tres problemas distintos en una sola columna:

| Qué aparece | Qué significa |
|---|---|
| `""` (vacío) | No se cargó |
| `"s/d"` | Se cargó explícitamente como "sin dato" |
| `"38 años"` | El número está, pero con texto pegado |

**Los tres tienen que terminar igual:** o un número entero, o vacío. Nunca cero — un
cero inventado se promedia y ensucia todo el análisis posterior.

Veamos también si hay valores absurdos:

In [ ]:
edades = pd.to_numeric(
    df["edad"].str.replace(" años", "", regex=False).str.strip(),
    errors="coerce",
)
edades.describe()

Hay una edad de **150** y una de **0**. Ninguna da error: son números válidos.
Solo se detectan si alguien decide cuál es el rango razonable.

> **El rango de validación es una decisión del dominio, no del código.** Para un
> padrón de beneficiarios, ¿corresponde 0–120 o 18–120?

---
## 4. La columna `telefono`

Acá está el problema más caro del archivo. Veamos en cuántos formatos viene cargado.

In [ ]:
def formato_telefono(telefono):
    """Clasifica el teléfono según cómo fue escrito."""
    if telefono.startswith("+54"):
        return "+54 area linea"
    if telefono.startswith("("):
        return "(0area) xxx-xxxx"
    if " 15" in telefono:
        return "0area 15x xxxxxx"
    return "todo pegado"


df["telefono"].apply(formato_telefono).value_counts()

Cuatro formatos distintos. Cualquier intento de partir el número por posición fija
—por ejemplo, "los primeros tres dígitos son la característica"— va a fallar en tres
de los cuatro.

**La forma correcta es al revés:** en lugar de suponer un largo, probamos contra una
lista de características que conocemos.

In [ ]:
# Las características que conocemos de arranque: las del NEA y CABA
AREAS_CONOCIDAS = {"11", "362", "370", "376", "379"}


def caracteristica(telefono):
    """Devuelve la característica del teléfono, o 'DESCONOCIDA'."""
    digitos = re.sub(r"\D", "", telefono)     # dejamos solo los números
    digitos = digitos.removeprefix("54")       # el código de país
    digitos = digitos.lstrip("0")              # el 0 de larga distancia

    for largo in (2, 3, 4):                    # las áreas tienen 2, 3 o 4 dígitos
        if digitos[:largo] in AREAS_CONOCIDAS:
            return digitos[:largo]
    return "DESCONOCIDA"


df["area_detectada"] = df["telefono"].apply(caracteristica)
df["area_detectada"].value_counts()

**Cuatro teléfonos que no reconocemos.** Veamos de dónde son:

In [ ]:
df.loc[df["area_detectada"] == "DESCONOCIDA",
       ["localidad", "provincia", "telefono"]]

Son de **Oberá (3755)** y **Clorinda (3718)** — dos localidades que no estaban en
nuestra lista inicial. Son características reales del plan de numeración argentino.

**Este es el resultado más importante de toda la exploración:** ahora sabemos que
`AREAS` en `config.py` tiene que incluir esas dos. Sin explorar primero, el ETL
habría roto cuatro teléfonos en silencio.

---
## 5. La columna `nombre`

In [ ]:
def formato_nombre(nombre):
    """Clasifica el nombre según cómo fue cargado."""
    if nombre != nombre.strip() or "  " in nombre.strip():
        return "espacios de más"
    if nombre.isupper():
        return "TODO MAYÚSCULA"
    if nombre.islower():
        return "todo minúscula"
    return "parece ok"


df["nombre"].apply(formato_nombre).value_counts()

Importa más de lo que parece: **`"Chaco"`, `"chaco "` y `"CHACO"` son tres claves
distintas.** Si agrupáramos sin normalizar antes, tendríamos tres grupos donde hay uno.

---
## 6. La columna `fecha_nac`

In [ ]:
def formato_fecha(fecha):
    """Clasifica la fecha según el formato en que fue escrita."""
    if re.fullmatch(r"\d{2}/\d{2}/\d{4}", fecha):
        return "dd/mm/aaaa"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", fecha):
        return "aaaa-mm-dd (ISO)"
    if re.fullmatch(r"\d{2}-\d{2}-\d{2}", fecha):
        return "dd-mm-aa"
    return "otro"


df["fecha_nac"].apply(formato_fecha).value_counts()

Los primeros dos formatos se convierten sin problema. El tercero es una trampa:

In [ ]:
from datetime import datetime

# El mismo formato, dos resultados muy distintos
print(datetime.strptime("25-03-65", "%d-%m-%y").date())   # 2065 !
print(datetime.strptime("25-03-69", "%d-%m-%y").date())   # 1969

Python corta en 68: de `00` a `68` asume años 2000, de `69` a `99` asume 1900. Una
persona nacida en **1965** queda registrada como nacida en **2065**, con edad negativa.

**No da error. Da un resultado absurdo.**

Con 27 filas así y sin ninguna regla que nos diga el siglo correcto, la decisión
honesta es dejar la fecha vacía. *Un dato faltante declarado es mejor que uno
inventado.*

---
## 7. ¿Hay filas repetidas?

El `documento` debería identificar a cada persona.

In [ ]:
repetidos = df[df.duplicated("documento", keep=False)]
repetidos[["documento", "nombre", "localidad", "provincia"]]

Un documento con dos nombres distintos. Alguien lo cargó mal. El ETL tiene que
quedarse con una sola fila y **dejar registrado que descartó la otra**.

---
## 8. Un vistazo a lo que sí se puede analizar

In [ ]:
fig, (izq, der) = plt.subplots(1, 2, figsize=(11, 4))

df["provincia"].value_counts().plot(kind="barh", ax=izq, color="#2E3192")
izq.set_title("Beneficiarios por provincia")

edades.dropna().plot(kind="hist", bins=20, ax=der, color="#2E3192")
der.set_title("Distribución de edades")
der.set_xlabel("edad")

plt.tight_layout()
plt.show()

El histograma muestra la cola en 150: el valor absurdo se ve a simple vista en un
gráfico, aunque no aparezca como error en ningún lado.

---
## Conclusiones: la lista de trabajo del ETL

| # | Problema encontrado | Qué tiene que hacer el ETL |
|---|---|---|
| 1 | `nombre` en minúscula, mayúscula y con espacios de más | Normalizar: espacios simples y mayúscula inicial |
| 2 | `edad` como `""`, `"s/d"` y `"38 años"` | Convertir a entero, o dejar vacío. Nunca cero |
| 3 | `edad` con valores de 0 y 150 | Descartar lo que esté fuera del rango acordado |
| 4 | `telefono` en cuatro formatos distintos | Separar en país, característica y línea |
| 5 | 4 teléfonos de Oberá y Clorinda | Agregar `3755` y `3718` a `AREAS` en `config.py` |
| 6 | `fecha_nac` en tres formatos, uno ambiguo | Pasar a ISO; dejar vacío el año de dos dígitos |
| 7 | Un `documento` repetido | Quedarse con una fila y registrar el descarte |

Con esta lista ya se puede escribir el ETL. **Este es el orden correcto: primero se
mira, después se programa.**

Todo lo que sigue está en:

```
src/extract.py     traer el dato tal como está
src/transform.py   las cuatro funciones de limpieza
src/load.py        validar, guardar y describir
src/main.py        llamar a las tres etapas en orden
```

Para correrlo, desde la raíz del proyecto:

```bash
python src/main.py
```

---
## Opcional: un reporte automático

Existen librerías que hacen buena parte de esta exploración solas.
[`ydata-profiling`](https://docs.profiling.ydata.ai/) genera un informe HTML completo
con tipos, faltantes, distribuciones y correlaciones, en una línea.

**No está en `requirements.txt` a propósito:** es una dependencia pesada y no
verificamos que funcione con la versión de pandas que usa el curso. Si querés
probarla, instalala aparte.

Sirve para tener un panorama rápido, pero no reemplaza esta exploración: el reporte
automático no sabe que `"s/d"` significa "sin dato" ni que `3718` es Clorinda. **Eso
lo sabe quien conoce los datos.**

In [ ]:
# Descomentá estas líneas para probarlo
# !pip install ydata-profiling

# from ydata_profiling import ProfileReport
# reporte = ProfileReport(df, title="Padrón de beneficiarios")
# reporte.to_file("../outputs/reporte_exploracion.html")